In [3]:
# real_data_test.py
"""
Test the matching engine on real MIMIC patients and trials.
"""

import json
import os
import pandas as pd
from matching_engine import (
    PatientState, Trial, Criterion,
    ICD10Hierarchy, compute_matching_indices,
    extract_criteria_stub  # For now, we'll use this as a test
)

def load_patient_states(cfg, num_patients=50):
    """Load real patient data from MIMIC."""
    diag_path = f"{cfg.OUTPUT_DIR}/diagnoses_clean.parquet"
    rx_path = f"{cfg.OUTPUT_DIR}/prescriptions_clean.parquet"
    labs_path = f"{cfg.OUTPUT_DIR}/labs_clean.parquet"
    
    diag_df = pd.read_parquet(diag_path)
    rx_df = pd.read_parquet(rx_path)
    labs_df = pd.read_parquet(labs_path)
    
    # Take a sample
    patient_ids = diag_df['SUBJECT_ID'].unique()[:num_patients]
    diag_df = diag_df[diag_df['SUBJECT_ID'].isin(patient_ids)]
    rx_df = rx_df[rx_df['SUBJECT_ID'].isin(patient_ids)]
    labs_df = labs_df[labs_df['SUBJECT_ID'].isin(patient_ids)]
    
    states = {}
    for pid in patient_ids:
        # Get diagnoses
        dx = set(diag_df[diag_df['SUBJECT_ID'] == pid]['ICD10_CODE'].astype(str))
        # Get medications
        rx = set(rx_df[rx_df['SUBJECT_ID'] == pid]['NDC'].astype(str))
        # Get lab values (latest)
        labs_pt = labs_df[labs_df['SUBJECT_ID'] == pid]
        lab_vals = {}
        if not labs_pt.empty:
            # Get latest value per lab
            latest = labs_pt.groupby('ITEMID').last()
            for itemid, row in latest.iterrows():
                if 'IMPUTED_VALUE_DECAYED' in row:
                    lab_vals[str(itemid)] = float(row['IMPUTED_VALUE_DECAYED'])
        
        states[pid] = PatientState(
            patient_id=str(pid),
            diagnosis_codes=dx,
            medication_codes=rx,
            lab_values=lab_vals
        )
    
    print(f"✅ Loaded {len(states)} patient states")
    return states

def load_trials_from_json(filepath):
    """Load trials from your structured JSON file."""
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    trials = []
    for item in data:
        inclusion = []
        exclusion = []
        for c in item.get('criteria', []):
            crit = Criterion(
                raw_entity=c.get('raw_entity', ''),
                entity_type=c.get('entity_type', 'diagnosis'),
                entity_code=c.get('entity_code', ''),
                operator=c.get('operator', 'EXISTS'),
                value=c.get('value'),
                is_inclusion=c.get('is_inclusion', True),
                severity_weight=c.get('severity_weight', 1.0),
                confidence=c.get('confidence', 1.0)
            )
            if crit.is_inclusion:
                inclusion.append(crit)
            else:
                exclusion.append(crit)
        
        trials.append(Trial(
            nct_id=item.get('nct_id', 'UNKNOWN'),
            inclusion_criteria=inclusion,
            exclusion_criteria=exclusion
        ))
    
    print(f"✅ Loaded {len(trials)} trials")
    return trials

def run_matching_test(patients, trials, hierarchy):
    """Test matching on real data."""
    print("\n" + "=" * 60)
    print("  REAL DATA MATCHING TEST")
    print("=" * 60)
    
    total_pairs = 0
    nonzero_inc = 0
    nonzero_exc = 0
    both_nonzero = 0
    
    # Track matches per trial
    trial_stats = {}
    
    for pid, patient in patients.items():
        for trial in trials:
            total_pairs += 1
            m_inc, m_exc = compute_matching_indices(patient, trial, hierarchy)
            
            if m_inc > 0:
                nonzero_inc += 1
            if m_exc > 0:
                nonzero_exc += 1
            if m_inc > 0 and m_exc > 0:
                both_nonzero += 1
            
            # Track per trial
            tid = trial.nct_id
            if tid not in trial_stats:
                trial_stats[tid] = {'inc': 0, 'exc': 0, 'pairs': 0}
            trial_stats[tid]['pairs'] += 1
            if m_inc > 0:
                trial_stats[tid]['inc'] += 1
            if m_exc > 0:
                trial_stats[tid]['exc'] += 1
    
    # Print results
    print(f"\n📊 Summary:")
    print(f"   Total patient-trial pairs: {total_pairs}")
    print(f"   Pairs with M_inc > 0: {nonzero_inc} ({nonzero_inc/total_pairs*100:.2f}%)")
    print(f"   Pairs with M_exc > 0: {nonzero_exc} ({nonzero_exc/total_pairs*100:.2f}%)")
    print(f"   Pairs with both > 0: {both_nonzero} ({both_nonzero/total_pairs*100:.2f}%)")
    
    if nonzero_inc == 0 and nonzero_exc == 0:
        print("\n❌ CRITICAL: No matches found! Pipeline will not learn.")
        print("\n🔍 Diagnosis: The trial codes don't match patient codes.")
        print("   - Patient codes are real ICD-10 codes (like I509)")
        print("   - Trial codes are still CODE_XXXX (placeholders)")
        print("\n💡 Solution: Run the code mapping script to replace CODE_XXXX")
        return False
    
    print("\n📊 Per-trial statistics:")
    for tid, stats in trial_stats.items():
        inc_pct = stats['inc'] / stats['pairs'] * 100
        exc_pct = stats['exc'] / stats['pairs'] * 100
        print(f"   {tid}: {stats['inc']}/{stats['pairs']} inc ({inc_pct:.1f}%), "
              f"{stats['exc']}/{stats['pairs']} exc ({exc_pct:.1f}%)")
    
    print("\n✅ Matches found! Pipeline can learn.")
    return True

def main():
    from config import Config
    cfg = Config()
    
    # 1. Load hierarchy
    print("📊 Loading hierarchy...")
    hierarchy_paths = [
        "icd10_hierarchy.csv",
        "../icd10_hierarchy.csv",
    ]
    hierarchy = None
    for path in hierarchy_paths:
        if os.path.exists(path):
            hierarchy = ICD10Hierarchy(path)
            print(f"   ✅ Loaded from {path}")
            break
    if hierarchy is None:
        print("   ⚠️ No hierarchy file found. Using exact-match only.")
        hierarchy = ICD10Hierarchy()
    
    # 2. Load patients
    print("\n📊 Loading patients...")
    patients = load_patient_states(cfg, num_patients=20)
    
    # 3. Load trials
    print("\n📊 Loading trials...")
    trial_paths = [
        f"{cfg.TRIALS_DATA_DIR}/structured_clinical_trials.json",
        "structured_clinical_trials.json"
    ]
    trials = []
    for path in trial_paths:
        if os.path.exists(path):
            trials = load_trials_from_json(path)
            break
    
    if not trials:
        print("❌ No trials found!")
        return
    
    # 4. Run test
    success = run_matching_test(patients, trials, hierarchy)
    
    # 5. If no matches, show sample codes for diagnosis
    if not success:
        print("\n🔍 Sample patient diagnosis codes:")
        for pid, patient in list(patients.items())[:3]:
            print(f"   Patient {pid}: {list(patient.diagnosis_codes)[:5]}")
        
        print("\n🔍 Sample trial inclusion codes:")
        for trial in trials[:2]:
            print(f"   Trial {trial.nct_id}:")
            for c in trial.inclusion_criteria[:3]:
                print(f"      → {c.entity_code} ({c.raw_entity[:30]})")

if __name__ == "__main__":
    main()

📊 Loading hierarchy...
   ✅ Loaded from icd10_hierarchy.csv

📊 Loading patients...
✅ Loaded 20 patient states

📊 Loading trials...
✅ Loaded 149 trials

  REAL DATA MATCHING TEST

📊 Summary:
   Total patient-trial pairs: 2980
   Pairs with M_inc > 0: 214 (7.18%)
   Pairs with M_exc > 0: 426 (14.30%)
   Pairs with both > 0: 104 (3.49%)

📊 Per-trial statistics:
   NCT07732361: 0/20 inc (0.0%), 0/20 exc (0.0%)
   NCT07732309: 0/20 inc (0.0%), 10/20 exc (50.0%)
   NCT07732283: 7/20 inc (35.0%), 0/20 exc (0.0%)
   NCT07732244: 0/20 inc (0.0%), 0/20 exc (0.0%)
   NCT07732231: 0/20 inc (0.0%), 0/20 exc (0.0%)
   NCT07732218: 4/20 inc (20.0%), 4/20 exc (20.0%)
   NCT07731997: 0/20 inc (0.0%), 2/20 exc (10.0%)
   NCT07731932: 0/20 inc (0.0%), 0/20 exc (0.0%)
   NCT07731724: 0/20 inc (0.0%), 0/20 exc (0.0%)
   NCT07731490: 0/20 inc (0.0%), 0/20 exc (0.0%)
   NCT07731464: 0/20 inc (0.0%), 0/20 exc (0.0%)
   NCT07731399: 4/20 inc (20.0%), 7/20 exc (35.0%)
   NCT07731334: 0/20 inc (0.0%), 0/20 exc (

In [1]:
# test_hierarchy.py
from matching_engine import ICD10Hierarchy

# Load hierarchy
h = ICD10Hierarchy("icd10_hierarchy.csv")

# Check if entries loaded
print(f"Loaded {len(h.child_to_parent)} parent-child relationships")

if len(h.child_to_parent) > 0:
    print("✅ Hierarchy has content!")
    print("Sample:", list(h.child_to_parent.items())[:3])
else:
    print("❌ Hierarchy is empty — fix the CSV file")

Loaded 0 parent-child relationships
❌ Hierarchy is empty — fix the CSV file
